In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display
from src import preprocessing as pp, feature_engineering as fe, pipeline as P
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
def show(name, w=800): display(Image(filename=str(FIG / name), width=w))

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 09 Proposed hybrid models
Soft voting, CV-weighted soft voting, stacking - all built from the tuned LR + RF + XGBoost. The hybrid is *tested*, not assumed better.

In [2]:
import json; print(json.load(open(OUT/'results.json'))['voting_weights'])
t = pd.read_csv(TAB/'model_comparison_A_characteristics.csv'); t[t.Model.str.startswith('Hybrid') | t.Model.isin(['Logistic Regression','Random Forest','XGBoost'])].round(3)

{'Logistic Regression': 0.3248687555856433, 'Random Forest': 0.33281576059869117, 'XGBoost': 0.3423154838156654}


,Model,Accuracy,Balanced Acc.,Precision (macro),Recall (macro),Specificity (macro),F1 (macro),F1 (weighted),ROC-AUC (OvR macro),PR-AUC (macro),Brier (multiclass),Low<->High errors (%),F1 (macro) 95% CI,CV macro-F1 (train)
0,Logistic Regression,0.387,0.387,0.384,0.387,0.693,0.384,0.384,0.580,0.408,0.655,17.778,"[0.321, 0.444]",0.398
2,Random Forest,0.391,0.391,0.372,0.391,0.696,0.375,0.375,0.591,0.427,0.645,18.222,"[0.312, 0.433]",0.408
3,XGBoost,0.396,0.396,0.391,0.396,0.698,0.393,0.393,0.568,0.404,0.682,16.000,"[0.333, 0.450]",0.419
5,Hybrid: Soft Voting,0.382,0.382,0.373,0.382,0.691,0.373,0.373,0.581,0.418,0.650,18.667,"[0.313, 0.435]",0.408
6,Hybrid: Weighted Voting,0.382,0.382,0.371,0.382,0.691,0.372,0.372,0.581,0.418,0.651,18.667,"[0.310, 0.431]",0.412
7,Hybrid: Stacking,0.387,0.387,0.348,0.387,0.693,0.356,0.356,0.594,0.431,0.641,18.667,"[0.301, 0.415]",0.379


### Set B (design 1 + held-out quality measures) and Design 2 (outcome index; care-process features)

In [3]:
pd.read_csv(TAB/'model_comparison_B_plus_heldout_quality.csv').round(3)

,Model,Accuracy,Balanced Acc.,Precision (macro),Recall (macro),Specificity (macro),F1 (macro),F1 (weighted),ROC-AUC (OvR macro),PR-AUC (macro),Brier (multiclass),Low<->High errors (%),F1 (macro) 95% CI,CV macro-F1 (train)
0,Logistic Regression,0.391,0.391,0.388,0.391,0.696,0.387,0.387,0.584,0.424,0.655,20.000,"[0.322, 0.451]",0.397
1,Decision Tree,0.364,0.364,0.355,0.364,0.682,0.350,0.350,0.539,0.381,0.737,23.111,"[0.290, 0.413]",0.378
2,Random Forest,0.404,0.404,0.400,0.404,0.702,0.398,0.398,0.609,0.459,0.637,18.667,"[0.332, 0.454]",0.426
3,XGBoost,0.400,0.400,0.401,0.400,0.700,0.394,0.394,0.597,0.443,0.643,16.444,"[0.331, 0.456]",0.413
4,Majority-class baseline,0.333,0.333,0.111,0.333,0.667,0.167,0.167,0.500,0.333,0.667,33.333,"[0.144, 0.189]",NaN
5,Hybrid: Soft Voting,0.409,0.409,0.404,0.409,0.704,0.400,0.400,0.600,0.459,0.638,19.111,"[0.335, 0.462]",0.407
6,Hybrid: Weighted Voting,0.409,0.409,0.404,0.409,0.704,0.400,0.400,0.601,0.459,0.638,19.111,"[0.335, 0.462]",0.405
7,Hybrid: Stacking,0.400,0.400,0.369,0.400,0.700,0.368,0.368,0.608,0.457,0.636,20.000,"[0.309, 0.427]",0.381


In [4]:
pd.read_csv(TAB/'model_comparison_C_outcome_index_process_features.csv').round(3)

,Model,Accuracy,Balanced Acc.,Precision (macro),Recall (macro),Specificity (macro),F1 (macro),F1 (weighted),ROC-AUC (OvR macro),PR-AUC (macro),Brier (multiclass),Low<->High errors (%),F1 (macro) 95% CI,CV macro-F1 (train)
0,Logistic Regression,0.404,0.404,0.401,0.404,0.702,0.403,0.403,0.591,0.432,0.654,14.667,"[0.339, 0.466]",0.406
1,Decision Tree,0.400,0.400,0.411,0.400,0.700,0.401,0.401,0.585,0.412,0.678,16.444,"[0.337, 0.465]",0.405
2,Random Forest,0.471,0.471,0.467,0.471,0.736,0.464,0.464,0.616,0.444,0.647,16.444,"[0.395, 0.532]",0.429
3,XGBoost,0.462,0.462,0.462,0.462,0.731,0.461,0.461,0.619,0.454,0.642,12.000,"[0.394, 0.527]",0.432
4,Majority-class baseline,0.333,0.333,0.111,0.333,0.667,0.167,0.167,0.500,0.333,0.667,33.333,"[0.144, 0.189]",NaN
5,Hybrid: Soft Voting,0.440,0.440,0.435,0.440,0.720,0.437,0.437,0.615,0.450,0.639,14.667,"[0.371, 0.497]",0.415
6,Hybrid: Weighted Voting,0.444,0.444,0.440,0.444,0.722,0.441,0.441,0.615,0.450,0.639,13.778,"[0.374, 0.502]",0.419
7,Hybrid: Stacking,0.462,0.462,0.441,0.462,0.731,0.439,0.439,0.626,0.449,0.637,15.111,"[0.374, 0.504]",0.392
